# Non - Textual Data PreProcessing

In [1]:

import pandas as pd

df = pd.read_csv("C:/MIMIC Structured dataset/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# convert to string (IMPORTANT)
df['icd_code'] = df['icd_code'].astype(str)

# cardiac arrest codes
ca_codes = ['4275', 'I46', 'I462', 'I468', 'I469']

# filter
df_ca = df[df['icd_code'].str.startswith(tuple(ca_codes))]

df_ca.head() 

,subject_id,hadm_id,seq_num,icd_code,icd_version
933,10001884,26184834,10,I469,10
6654,10009823,22096005,13,I469,10
7124,10010471,29842315,3,I462,10
8929,10013569,27993048,2,4275,9
13850,10021454,29492087,1,I469,10


In [2]:
df_ca

,subject_id,hadm_id,seq_num,icd_code,icd_version
933,10001884,26184834,10,I469,10
6654,10009823,22096005,13,I469,10
7124,10010471,29842315,3,I462,10
8929,10013569,27993048,2,4275,9
13850,10021454,29492087,1,I469,10
...,...,...,...,...,...
6352563,19983257,21588174,14,4275,9
6352835,19984052,25784208,5,I469,10
6357795,19990427,29695607,21,I469,10
6359705,19993336,23077223,2,I468,10


In [3]:
#Merging ICU stays 

In [4]:
icu = pd.read_csv("C:/MIMIC Structured dataset/mimic-iv-3.1/icu/icustays.csv")

df_cohort = df_ca.merge(icu, on=['subject_id', 'hadm_id'])

In [5]:
#Add demographics

In [6]:
patients = pd.read_csv("C:/MIMIC Structured dataset/mimic-iv-3.1/hosp/patients.csv")

df_cohort = df_cohort.merge(patients, on='subject_id')

In [7]:
admissions = pd.read_csv("C:/MIMIC Structured dataset/mimic-iv-3.1/hosp/admissions.csv")

df_cohort = df_cohort.merge(admissions[['hadm_id','deathtime','dischtime']], on='hadm_id')

df_cohort['mortality'] = (df_cohort['deathtime'] <= df_cohort['dischtime']).astype(int)

In [8]:
df_cohort.shape

(2896, 19)

In [9]:
df_cohort.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,deathtime,dischtime,mortality
0,10001884,26184834,10,I469,10,37510196,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2131-01-11 04:20:05,2131-01-20 08:27:30,9.171817,F,68,2122,2008 - 2010,2131-01-20,2131-01-20 05:15:00,2131-01-20 05:15:00,1
1,10009823,22096005,13,I469,10,31805686,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),2150-12-06 06:46:10,2150-12-06 20:33:52,0.574792,F,45,2150,2020 - 2022,2150-12-06,2150-12-06 14:22:00,2150-12-06 14:22:00,1
2,10010471,29842315,3,I462,10,32119961,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2155-12-02 20:33:00,2155-12-07 18:19:18,4.907153,F,89,2155,2014 - 2016,2155-12-07,2155-12-07 15:30:00,2155-12-07 15:30:00,1
3,10013569,27993048,2,4275,9,38857852,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2167-11-26 16:26:07,2167-12-05 16:53:53,9.019282,F,54,2165,2008 - 2010,NaN,NaN,2167-12-25 14:53:00,0
4,10013569,27993048,2,4275,9,39673498,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2167-12-10 18:23:25,2167-12-18 18:16:14,7.995012,F,54,2165,2008 - 2010,NaN,NaN,2167-12-25 14:53:00,0


In [10]:
#Removing the duplicate icu stays and keeping only first icu stays

In [11]:
df_cohort = df_cohort.sort_values(by=['subject_id','intime'])
df_cohort = df_cohort.groupby('subject_id').first().reset_index()

In [12]:
# loading the chartevents datasets (40gb wala) 

In [13]:
import pandas as pd

stay_ids = set(df_cohort['stay_id'])

filtered_chunks = []

chunks = pd.read_csv(
    "C:/MIMIC Structured dataset/mimic-iv-3.1/icu/chartevents.csv",
    usecols=['stay_id','itemid','charttime','valuenum'],
    chunksize=100000
)

for i, chunk in enumerate(chunks):
    chunk = chunk[chunk['stay_id'].isin(stay_ids)]
    filtered_chunks.append(chunk)

    if i % 10 == 0:
        print(f"Processed {i} chunks")

df_chartevents = pd.concat(filtered_chunks, ignore_index=True)

df_chartevents.shape

Processed 0 chunks
Processed 10 chunks
Processed 20 chunks
Processed 30 chunks
Processed 40 chunks
Processed 50 chunks
Processed 60 chunks
Processed 70 chunks
Processed 80 chunks
Processed 90 chunks
Processed 100 chunks
Processed 110 chunks
Processed 120 chunks
Processed 130 chunks
Processed 140 chunks
Processed 150 chunks
Processed 160 chunks
Processed 170 chunks
Processed 180 chunks
Processed 190 chunks
Processed 200 chunks
Processed 210 chunks
Processed 220 chunks
Processed 230 chunks
Processed 240 chunks
Processed 250 chunks
Processed 260 chunks
Processed 270 chunks
Processed 280 chunks
Processed 290 chunks
Processed 300 chunks
Processed 310 chunks
Processed 320 chunks
Processed 330 chunks
Processed 340 chunks
Processed 350 chunks
Processed 360 chunks
Processed 370 chunks
Processed 380 chunks
Processed 390 chunks
Processed 400 chunks
Processed 410 chunks
Processed 420 chunks
Processed 430 chunks
Processed 440 chunks
Processed 450 chunks
Processed 460 chunks
Processed 470 chunks
Pro

(18532890, 4)

In [14]:
df_chartevents.head()

,stay_id,charttime,itemid,valuenum
0,37510196,2131-01-18 04:19:00,227944,NaN
1,37510196,2131-01-18 04:19:00,227945,NaN
2,37510196,2131-01-18 04:19:00,227946,NaN
3,37510196,2131-01-18 04:19:00,227947,NaN
4,37510196,2131-01-18 04:19:00,227948,NaN


In [15]:
df_chartevents['itemid'].value_counts().head(20)

itemid
227969    428271
220045    351084
220210    346407
220277    338111
220048    316085
224650    271715
227958    210517
220181    171893
220179    171853
220180    171827
220052    166504
220050    165665
220051    165619
228928    161977
228905    149883
224080    131076
224082    129716
224093    124935
224086    122291
228924    111361
Name: count, dtype: int64

In [16]:
# from this chartevents file we will extract the vital sign like  heart rate (HR), systolic/diastolic/mean
#blood pressure (SBP, DBP, MBP), respiratory rate (RR), body temperature (BT), and
#oxygen saturation (SpO2)

In [17]:
#filtering important vitals

vital_itemids = [
    220045,  # Heart Rate
    220210,  # Respiratory Rate
    220277,  # SpO2
    220179,  # SBP
    220180,  # DBP
    220181   # MBP
]

df_vitals = df_chartevents[df_chartevents['itemid'].isin(vital_itemids)]

In [18]:
# Removing NaN wala rows (if the value is a string like text data then it will show NaN and if it is numeric value then it will show the respective numbeer

In [19]:
df_vitals = df_vitals.dropna(subset=['valuenum'])

In [20]:
df_vitals.head()

,stay_id,charttime,itemid,valuenum
34,37510196,2131-01-11 06:00:00,220045,72.0
36,37510196,2131-01-11 06:00:00,220210,20.0
37,37510196,2131-01-11 06:00:00,220277,100.0
38,37510196,2131-01-11 06:01:00,220179,102.0
39,37510196,2131-01-11 06:01:00,220180,63.0


In [21]:
# here we are using vital signs for only 24 hours 

In [22]:
# converting to datetime
df_vitals['charttime'] = pd.to_datetime(df_vitals['charttime'])
df_cohort['intime'] = pd.to_datetime(df_cohort['intime'])

# merging intime
df_vitals = df_vitals.merge(
    df_cohort[['stay_id','intime']],
    on='stay_id',
    how='left'
)

# keeping only first 24 hours
df_vitals = df_vitals[
    (df_vitals['charttime'] >= df_vitals['intime']) &
    (df_vitals['charttime'] <= df_vitals['intime'] + pd.Timedelta(hours=24))
]

In [23]:
# creating features i.e mean, min, and max

In [24]:
df_vitals_agg = df_vitals.groupby(['stay_id','itemid'])['valuenum'].agg(['mean','min','max']).reset_index()

In [25]:
#converting row to columns

In [26]:
df_vitals_pivot = df_vitals_agg.pivot(index='stay_id', columns='itemid')

# flatten column names
df_vitals_pivot.columns = [
    f"{col[1]}_{col[0]}" for col in df_vitals_pivot.columns
]

df_vitals_pivot = df_vitals_pivot.reset_index()

In [27]:
rename_dict = {
    '220045_mean': 'hr_mean',
    '220045_min': 'hr_min',
    '220045_max': 'hr_max',

    '220210_mean': 'rr_mean',
    '220210_min': 'rr_min',
    '220210_max': 'rr_max',

    '220277_mean': 'spo2_mean',
    '220277_min': 'spo2_min',
    '220277_max': 'spo2_max',

    '220179_mean': 'sbp_mean',
    '220179_min': 'sbp_min',
    '220179_max': 'sbp_max',

    '220180_mean': 'dbp_mean',
    '220180_min': 'dbp_min',
    '220180_max': 'dbp_max',

    '220181_mean': 'mbp_mean',
    '220181_min': 'mbp_min',
    '220181_max': 'mbp_max',
}

df_vitals_pivot = df_vitals_pivot.rename(columns=rename_dict)

In [28]:
df_final = df_cohort.merge(df_vitals_pivot, on='stay_id', how='left')

In [29]:
df_final.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version,stay_id,first_careunit,last_careunit,intime,outtime,...,dbp_min,mbp_min,rr_min,spo2_min,hr_max,sbp_max,dbp_max,mbp_max,rr_max,spo2_max
0,10001884,26184834,10,I469,10,37510196,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2131-01-11 04:20:05,2131-01-20 08:27:30,...,12.0,46.0,10.0,89.0,80.0,180.0,123.0,130.0,26.0,100.0
1,10009823,22096005,13,I469,10,31805686,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),2150-12-06 06:46:10,2150-12-06 20:33:52,...,38.0,43.0,12.0,95.0,113.0,145.0,109.0,121.0,30.0,99.0
2,10010471,29842315,3,I462,10,32119961,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2155-12-02 20:33:00,2155-12-07 18:19:18,...,42.0,52.0,9.0,93.0,96.0,139.0,79.0,92.0,24.0,100.0
3,10013569,27993048,2,4275,9,38857852,Coronary Care Unit (CCU),Coronary Care Unit (CCU),2167-11-26 16:26:07,2167-12-05 16:53:53,...,47.0,59.0,10.0,89.0,93.0,126.0,87.0,91.0,24.0,100.0
4,10021454,29492087,1,I469,10,33300154,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2125-03-17 02:41:26,2125-03-24 20:18:57,...,NaN,NaN,19.0,99.0,83.0,NaN,NaN,NaN,23.0,100.0


In [30]:
df_final.shape

(2307, 37)

In [31]:
df_final.columns

Index(['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version',
       'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime',
       'los', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group',
       'dod', 'deathtime', 'dischtime', 'mortality', 'hr_mean', 'sbp_mean',
       'dbp_mean', 'mbp_mean', 'rr_mean', 'spo2_mean', 'hr_min', 'sbp_min',
       'dbp_min', 'mbp_min', 'rr_min', 'spo2_min', 'hr_max', 'sbp_max',
       'dbp_max', 'mbp_max', 'rr_max', 'spo2_max'],
      dtype='object')

In [32]:
# Adding body temperature in the dataset (BT)
temp_items = [223761, 678]   # Fahrenheit + Celsius   #select temperature itemids
df_temp = df_chartevents[df_chartevents['itemid'].isin(temp_items)]       #FILTER FROM CHARTEVENTS
df_temp = df_temp[df_temp['stay_id'].isin(df_final['stay_id'])]         #KEEP ONLY YOUR COHORT
df_temp['charttime'] = pd.to_datetime(df_temp['charttime'])          #CONVERT TO DATETIME
df_temp = df_temp.merge(                                             #MERGE INTIME (FOR 24h FILTER)
    df_final[['stay_id','intime']],
    on='stay_id',
    how='left'
)
df_temp = df_temp[                                                    #FILTER FIRST 24 HOURS
    (df_temp['charttime'] >= df_temp['intime']) &
    (df_temp['charttime'] <= df_temp['intime'] + pd.Timedelta(hours=24))
]
# Convert Celsius → Fahrenheit                                        #UNIT CONVERSION
df_temp.loc[df_temp['valuenum'] < 50, 'valuenum'] = (
    df_temp['valuenum'] * 9/5 + 32
)
df_temp_agg = df_temp.groupby('stay_id')['valuenum'].agg(['mean','min','max']).reset_index()       #AGGREGATE

df_temp_agg.columns = ['stay_id','bt_mean','bt_min','bt_max']

df_final = df_final.merge(df_temp_agg, on='stay_id', how='left')             #MERGE INTO FINAL DATA

df_final[['bt_mean']].head()                                    

,bt_mean
0,NaN
1,97.533333
2,97.880000
3,98.400000
4,96.500000


In [33]:
# Now we come to cleaning and Aggregration part
# Cleaning the data 
# HR
df_final.loc[(df_final['hr_mean'] < 30) | (df_final['hr_mean'] > 220), 'hr_mean'] = None

# SBP
df_final.loc[(df_final['sbp_mean'] < 50) | (df_final['sbp_mean'] > 250), 'sbp_mean'] = None

# DBP
df_final.loc[(df_final['dbp_mean'] < 30) | (df_final['dbp_mean'] > 150), 'dbp_mean'] = None

# RR
df_final.loc[(df_final['rr_mean'] < 5) | (df_final['rr_mean'] > 60), 'rr_mean'] = None

# SpO2
df_final.loc[(df_final['spo2_mean'] < 50) | (df_final['spo2_mean'] > 100), 'spo2_mean'] = None

# Body Temperature 
df_final.loc[(df_final['bt_mean'] < 80) | (df_final['bt_mean'] > 110), 'bt_mean'] = None

In [34]:
# Select only vital columns
vital_cols = [
    'hr_mean','sbp_mean','dbp_mean','mbp_mean',
    'rr_mean','spo2_mean','bt_mean'
]

# Calculate missing count and percentage
missing_vitals = pd.DataFrame({
    'Missing Count': df_final[vital_cols].isna().sum(),
    'Missing %': (df_final[vital_cols].isna().mean() * 100).round(2)
})

missing_vitals = missing_vitals.reset_index()
missing_vitals.columns = ['Vital Sign','Missing Count','Missing %']

missing_vitals

,Vital Sign,Missing Count,Missing %
0,hr_mean,8,0.35
1,sbp_mean,304,13.18
2,dbp_mean,314,13.61
3,mbp_mean,296,12.83
4,rr_mean,19,0.82
5,spo2_mean,43,1.86
6,bt_mean,621,26.92


In [35]:
# HANDLE MISSING VALUES (INITIAL)
# Mean imputation (low missing)
df_final['hr_mean'] = df_final['hr_mean'].fillna(df_final['hr_mean'].mean())
df_final['dbp_mean'] = df_final['dbp_mean'].fillna(df_final['dbp_mean'].mean())

# Median imputation (skewed)
df_final['sbp_mean'] = df_final['sbp_mean'].fillna(df_final['sbp_mean'].median())
df_final['mbp_mean'] = df_final['mbp_mean'].fillna(df_final['mbp_mean'].median())
df_final['rr_mean'] = df_final['rr_mean'].fillna(df_final['rr_mean'].median())
df_final['spo2_mean'] = df_final['spo2_mean'].fillna(df_final['spo2_mean'].median())

In [36]:
#MBP HARMONIZATION
df_final['mbp_mean'] = df_final['mbp_mean'].fillna(
    (df_final['sbp_mean'] + 2 * df_final['dbp_mean']) / 3
)

In [37]:
df_final.isna().sum()

subject_id             0
hadm_id                0
seq_num                0
icd_code               0
icd_version            0
stay_id                0
first_careunit         0
last_careunit          0
intime                 0
outtime                2
los                    2
gender                 0
anchor_age             0
anchor_year            0
anchor_year_group      0
dod                  638
deathtime            996
dischtime              0
mortality              0
hr_mean                0
sbp_mean               0
dbp_mean               0
mbp_mean               0
rr_mean                0
spo2_mean              0
hr_min                 5
sbp_min              296
dbp_min              296
mbp_min              296
rr_min                15
spo2_min              37
hr_max                 5
sbp_max              296
dbp_max              296
mbp_max              296
rr_max                15
spo2_max              37
bt_mean              621
bt_min               619
bt_max               619


In [38]:
df_final.columns

Index(['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version',
       'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime',
       'los', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group',
       'dod', 'deathtime', 'dischtime', 'mortality', 'hr_mean', 'sbp_mean',
       'dbp_mean', 'mbp_mean', 'rr_mean', 'spo2_mean', 'hr_min', 'sbp_min',
       'dbp_min', 'mbp_min', 'rr_min', 'spo2_min', 'hr_max', 'sbp_max',
       'dbp_max', 'mbp_max', 'rr_max', 'spo2_max', 'bt_mean', 'bt_min',
       'bt_max'],
      dtype='object')

In [39]:
#Loading the lab events

lab_chunks = pd.read_csv(
    "C:/MIMIC Structured dataset/mimic-iv-3.1/hosp/labevents.csv",
    usecols=['hadm_id','itemid','charttime','valuenum'],
    chunksize=100000
)

hadm_ids = set(df_final['hadm_id'])

filtered_labs = []

for i, chunk in enumerate(lab_chunks):
    chunk = chunk[chunk['hadm_id'].isin(hadm_ids)]
    filtered_labs.append(chunk)

    if i % 10 == 0:
        print(f"Processed {i} chunks")

df_labs = pd.concat(filtered_labs, ignore_index=True)

Processed 0 chunks
Processed 10 chunks
Processed 20 chunks
Processed 30 chunks
Processed 40 chunks
Processed 50 chunks
Processed 60 chunks
Processed 70 chunks
Processed 80 chunks
Processed 90 chunks
Processed 100 chunks
Processed 110 chunks
Processed 120 chunks
Processed 130 chunks
Processed 140 chunks
Processed 150 chunks
Processed 160 chunks
Processed 170 chunks
Processed 180 chunks
Processed 190 chunks
Processed 200 chunks
Processed 210 chunks
Processed 220 chunks
Processed 230 chunks
Processed 240 chunks
Processed 250 chunks
Processed 260 chunks
Processed 270 chunks
Processed 280 chunks
Processed 290 chunks
Processed 300 chunks
Processed 310 chunks
Processed 320 chunks
Processed 330 chunks
Processed 340 chunks
Processed 350 chunks
Processed 360 chunks
Processed 370 chunks
Processed 380 chunks
Processed 390 chunks
Processed 400 chunks
Processed 410 chunks
Processed 420 chunks
Processed 430 chunks
Processed 440 chunks
Processed 450 chunks
Processed 460 chunks
Processed 470 chunks
Pro

In [40]:
# selecting the important lab events
lab_items = {
    # Hemoglobin
    50811: 'hemoglobin',
    51222: 'hemoglobin',

    # Hematocrit
    50810: 'hematocrit',
    51221: 'hematocrit',

    # Others
    51265: 'platelet',
    51301: 'wbc',
    51274: 'pt',
    51275: 'inr',
    50912: 'creatinine',
    51006: 'bun',
    50931: 'glucose',
    50971: 'potassium',
    50983: 'sodium',
    50893: 'calcium',
    50902: 'chloride',
    50868: 'anion_gap',
    50882: 'bicarbonate',
    50813: 'lactate',
    50820: 'ph'
}

df_labs = df_labs[df_labs['itemid'].isin(lab_items.keys())]  #filter

In [41]:
df_labs['lab_name'] = df_labs['itemid'].map(lab_items)

In [42]:
# Removing the outliers

df_labs.loc[(df_labs['valuenum'] < 0), 'valuenum'] = None
df_labs.loc[(df_labs['valuenum'] > 600), 'valuenum'] = None  # glucose

In [43]:
# Now aggregating the values of lab events 
df_labs_agg = df_labs.groupby(['hadm_id','lab_name'])['valuenum'].agg(['mean','min','max']).reset_index()

In [44]:
# pivot
df_labs_pivot = df_labs_agg.pivot(index='hadm_id', columns='lab_name')

df_labs_pivot.columns = [
    f"{col[1]}_{col[0]}" for col in df_labs_pivot.columns
]

df_labs_pivot = df_labs_pivot.reset_index()

In [45]:
# merging 
df_final = df_final.merge(df_labs_pivot, on='hadm_id', how='left')

In [46]:
# missing values table for lab events
lab_cols = [
    'hemoglobin_mean','hematocrit_mean','platelet_mean','wbc_mean',
    'pt_mean','inr_mean','creatinine_mean','bun_mean','glucose_mean',
    'potassium_mean','sodium_mean','calcium_mean','chloride_mean',
    'anion_gap_mean','bicarbonate_mean','lactate_mean','ph_mean'
]

missing_labs = pd.DataFrame({
    'Missing Count': df_final[lab_cols].isna().sum(),
    'Missing %': (df_final[lab_cols].isna().mean() * 100).round(2)
})

missing_labs = missing_labs.reset_index()
missing_labs.columns = ['Lab Test','Missing Count','Missing %']

missing_labs

,Lab Test,Missing Count,Missing %
0,hemoglobin_mean,70,3.03
1,hematocrit_mean,69,2.99
2,platelet_mean,75,3.25
3,wbc_mean,71,3.08
4,pt_mean,104,4.51
5,inr_mean,108,4.68
6,creatinine_mean,70,3.03
7,bun_mean,70,3.03
8,glucose_mean,79,3.42
9,potassium_mean,71,3.08


In [47]:
# Handling the missing values (Lab events)

#Mean imputation (<5%)

mean_cols = missing_labs[missing_labs['Missing %'] < 5]['Lab Test']

for col in mean_cols:
    df_final[col].fillna(df_final[col].mean(), inplace=True)


# Median Imputation (5-10%)
median_cols = missing_labs[
    (missing_labs['Missing %'] >= 5) & (missing_labs['Missing %'] <= 10)
]['Lab Test']

for col in median_cols:
    df_final[col].fillna(df_final[col].median(), inplace=True)

C:\Users\amang\AppData\Local\Temp\ipykernel_14580\3791741155.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_final[col].fillna(df_final[col].mean(), inplace=True)
C:\Users\amang\AppData\Local\Temp\ipykernel_14580\3791741155.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

In [48]:
# Comorbidities 
df_diag = pd.read_csv("C:/MIMIC Structured dataset/mimic-iv-3.1/hosp/diagnoses_icd.csv")  #LOAD FULL DIAGNOSES DATA

df_diag = df_diag[df_diag['hadm_id'].isin(df_final['hadm_id'])]               #FILTER ONLY YOUR COHORT

df_diag['icd_code'] = df_diag['icd_code'].astype(str)                  #PREPARE ICD CODES

df_diag_grouped = df_diag.groupby('hadm_id')['icd_code'].apply(list).reset_index()           #GROUP ALL DIAGNOSES PER PATIENT

def has_prefix(code_list, prefix):                              #CREATE HELPER FUNCTION
    return any(code.startswith(prefix) for code in code_list)

#CREATE COMORBIDITY FLAGS
df_diag_grouped['copd'] = df_diag_grouped['icd_code'].apply(lambda x: has_prefix(x, 'J44')).astype(int)

df_diag_grouped['heart_failure'] = df_diag_grouped['icd_code'].apply(lambda x: has_prefix(x, 'I50')).astype(int)

df_diag_grouped['diabetes'] = df_diag_grouped['icd_code'].apply(lambda x: has_prefix(x, 'E11')).astype(int)

df_diag_grouped['hypertension'] = df_diag_grouped['icd_code'].apply(lambda x: has_prefix(x, 'I10')).astype(int)

df_comorb = df_diag_grouped[['hadm_id','copd','heart_failure','diabetes','hypertension']]   #KEEP ONLY REQUIRED COLUMNS

df_final = df_final.drop(columns=['copd','heart_failure','diabetes','hypertension'], errors='ignore')    #First remove old incorrect columns

df_final = df_final.merge(df_comorb, on='hadm_id', how='left')   #MERGE INTO FINAL DATASET

df_final[['copd','heart_failure','diabetes','hypertension']] = df_final[      #HANDLE MISSING
    ['copd','heart_failure','diabetes','hypertension']
].fillna(0)

df_final[['copd','heart_failure','diabetes','hypertension']].sum()

copd             151
heart_failure    513
diabetes         400
hypertension     336
dtype: int64

In [49]:
# Treatment Indicator Section
import pandas as pd

# Load inputevents
df_input = pd.read_csv("C:/MIMIC Structured dataset/mimic-iv-3.1/icu/inputevents.csv")

# Keep only cohort
df_input = df_input[df_input['stay_id'].isin(df_final['stay_id'])]

# Merge intime for 24h filter
df_input = df_input.merge(
    df_final[['stay_id','intime']],
    on='stay_id',
    how='left'
)

# Convert datetime
df_input['starttime'] = pd.to_datetime(df_input['starttime'])
df_input['intime'] = pd.to_datetime(df_input['intime'])

# Apply 24-hour filter
df_input = df_input[
    (df_input['starttime'] >= df_input['intime']) &
    (df_input['starttime'] <= df_input['intime'] + pd.Timedelta(hours=24))
]

# Define drug itemids
epinephrine_ids = [221289]
dopamine_ids = [221662]

# Create drug flags
df_input['epinephrine'] = df_input['itemid'].isin(epinephrine_ids).astype(int)
df_input['dopamine'] = df_input['itemid'].isin(dopamine_ids).astype(int)

# Aggregate per patient
df_treat = df_input.groupby('stay_id')[['epinephrine','dopamine']].max().reset_index()

# Ventilation (from chartevents)
vent_items = [224687, 224688]

df_vent = df_chartevents[df_chartevents['itemid'].isin(vent_items)]

# Merge intime
df_vent = df_vent.merge(
    df_final[['stay_id','intime']],
    on='stay_id',
    how='left'
)

# Convert datetime
df_vent['charttime'] = pd.to_datetime(df_vent['charttime'])
df_vent['intime'] = pd.to_datetime(df_vent['intime'])

# Apply 24-hour filter
df_vent = df_vent[
    (df_vent['charttime'] >= df_vent['intime']) &
    (df_vent['charttime'] <= df_vent['intime'] + pd.Timedelta(hours=24))
]

# Create ventilation flag
df_vent['received_ventilation'] = 1

df_vent = df_vent.groupby('stay_id')['received_ventilation'].max().reset_index()

# Merge into final dataset
df_final = df_final.drop(columns=['epinephrine','dopamine','received_ventilation'], errors='ignore')

df_final = df_final.merge(df_treat, on='stay_id', how='left')
df_final = df_final.merge(df_vent, on='stay_id', how='left')

# Fill missing
df_final[['epinephrine','dopamine','received_ventilation']] = df_final[
    ['epinephrine','dopamine','received_ventilation']
].fillna(0)

# Final check
print(df_final[['epinephrine','dopamine','received_ventilation']].mean()*100)

epinephrine             14.824447
dopamine                 9.926311
received_ventilation    73.992198
dtype: float64


In [50]:
# GCS Score

gcs_items = [223900, 223901, 220739]

df_gcs = df_chartevents[df_chartevents['itemid'].isin(gcs_items)]

df_gcs_agg = df_gcs.groupby(['stay_id','itemid'])['valuenum'].mean().reset_index()

df_gcs_pivot = df_gcs_agg.pivot(index='stay_id', columns='itemid', values='valuenum')

df_gcs_pivot.columns = ['gcs_eye','gcs_verbal','gcs_motor']

df_gcs_pivot['gcs_total'] = df_gcs_pivot.sum(axis=1)

df_gcs_pivot = df_gcs_pivot.reset_index()

df_final = df_final.merge(df_gcs_pivot, on='stay_id', how='left')

In [51]:
# Final imputation 

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Remove non-numeric columns first
cols_to_drop = [
    'subject_id','hadm_id','stay_id',
    'mortality',   # VERY IMPORTANT
    'intime','outtime','dischtime','deathtime','dod'
]

df_model = df_final.drop(columns=cols_to_drop, errors='ignore')
df_model = df_model.select_dtypes(include=['float64','int64'])

imputer = IterativeImputer(max_iter=5, random_state=0)

df_model_imputed = pd.DataFrame(
    imputer.fit_transform(df_model),
    columns=df_model.columns
)

C:\Users\amang\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:825: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [52]:
df_final[df_model.columns] = df_model_imputed

In [53]:
df_final.shape

(2307, 102)

In [54]:
df_final.isna().sum().sort_values(ascending = False)

deathtime     996
dod           638
outtime         2
subject_id      0
inr_min         0
             ... 
dbp_max         0
sbp_max         0
hr_max          0
spo2_min        0
gcs_total       0
Length: 102, dtype: int64

In [55]:
df_final.columns

Index(['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version',
       'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime',
       ...
       'heart_failure', 'diabetes', 'hypertension', 'epinephrine', 'dopamine',
       'received_ventilation', 'gcs_eye', 'gcs_verbal', 'gcs_motor',
       'gcs_total'],
      dtype='object', length=102)

In [57]:
# df_final.to_excel(
#     "C:/Users/amang/mimic_final_dataset1.xlsx",
#     index=False,
#     engine='openpyxl'
# )

In [59]:
df_final.shape

(2307, 102)